# 🛠️ LoRA Fine-Tuning & Evaluation mit Hugging Face, PEFT & WandB

Dieses Notebook führt dich durch den gesamten Prozess des **Supervised Fine-Tunings (SFT)** mittels **Hugging Face Transformers** und **PEFT (LoRA)**:
1. **Voraussetzungsprüfung:** Validierung von CUDA, VRAM und Konfigurationen.
2. **LoRA-Training:** Laden des Basismodells, Einrichten der PEFT-Adapter und Durchführung des Trainings.
3. **Lokale Evaluation:** Klassifikations-Report und Konfusionsmatrix.
4. **WandB-Evaluation (Neu):** Zusätzlicher, dedizierter Weights & Biases Run inklusive interaktiver **W&B Table** zur visuellen Überprüfung der Modellantworten.

In [1]:
# Import aller benötigten Bibliotheken für Training, PEFT, Tokenisierung, Logging und Metriken
import os
import sys
import torch
import logging
import json
from omegaconf import DictConfig, OmegaConf
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import get_peft_model, LoraConfig
from huggingface_hub import snapshot_download
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix
import wandb

# Logging konfigurieren für übersichtliche Statusmeldungen
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

PROJECT_ROOT = "/data/nemo-fraud-detection"
print("✅ Bibliotheken erfolgreich geladen.")

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-23 17:03:36.477845152 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


✅ Bibliotheken erfolgreich geladen.


### 🔍 Schritt 1: Voraussetzungen prüfen
Überprüft, ob eine GPU bereit steht, genügend VRAM vorhanden ist und die Konfigurationsdatei alle notwendigen Pfade enthält.

In [2]:
def validate_prerequisites(cfg: DictConfig) -> None:
    logging.info("🔍 Überprüfe Voraussetzungen für das Training...")

    if not torch.cuda.is_available():
        raise RuntimeError("❌ Keine CUDA-fähige GPU gefunden! Training erfordert eine GPU.")
    
    vram_free_gb = torch.cuda.mem_get_info()[0] / (1024**3)
    logging.info(f"✅ GPU erkannt: {torch.cuda.get_device_name(0)} (Freier VRAM: {vram_free_gb:.2f} GB)")

    model_path = cfg.model.get("restore_from_path", None)
    if not model_path:
        raise ValueError("❌ In der Konfiguration fehlt 'cfg.model.restore_from_path'!")

    logging.info("✅ Voraussetzungen erfolgreich geprüft.")

### Schritt 2: LoRA Fine-Tuning Pipeline
Lädt das Modell, wendet die PEFT-Konfiguration an, tokenisiert die Datensätze und startet den Hugging Face `Trainer`.

In [3]:
def run_training_and_evaluation(cfg: DictConfig) -> bool:
    try:
        validate_prerequisites(cfg)

        model_path = cfg.model.restore_from_path
        if model_path.startswith("hf://"):
            model_path = model_path.replace("hf://", "")

        if not os.path.exists(model_path) and not "/" in model_path:
            logging.info(f"📥 Lade Hugging Face Modell '{model_path}' herunter...")
            model_path = snapshot_download(repo_id=model_path)

        logging.info(f"📥 Initialisiere Standard 16-bit LoRA Modell von: {model_path}...")

        # 1. Basismodell im 16-bit Modus laden
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.float16,
            device_map="auto"
        )

        tokenizer = AutoTokenizer.from_pretrained(model_path)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"

        # 2. LoRA-Konfiguration (PEFT) definieren
        peft_config = LoraConfig(
            r=cfg.model.get("peft", {}).get("qlora_tuning", {}).get("adapter_dim", 16),
            lora_alpha=cfg.model.get("peft", {}).get("qlora_tuning", {}).get("lora_alpha", 32),
            target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc_in", "fc_out", "w1", "w2"],
            lora_dropout=cfg.model.get("peft", {}).get("qlora_tuning", {}).get("adapter_dropout", 0.05),
            bias="none",
            task_type="CAUSAL_LM"
        )

        model = get_peft_model(model, peft_config)
        model.print_trainable_parameters()

        # 3. Datensätze laden
        train_file = OmegaConf.to_container(cfg.model.data.train_ds.file_names, resolve=True)
        if isinstance(train_file, list):
            train_file = train_file[0]
        if not os.path.isabs(train_file):
            train_file = os.path.join(PROJECT_ROOT, train_file)

        val_file_cfg = cfg.model.data.get("validation_ds", {}).get("file_names", None)
        val_file = None
        if val_file_cfg:
            val_file = OmegaConf.to_container(val_file_cfg, resolve=True)
            if isinstance(val_file, list):
                val_file = val_file[0]
            if not os.path.isabs(val_file):
                val_file = os.path.join(PROJECT_ROOT, val_file)

        logging.info(f"📂 Lade Trainingsdaten von: {train_file}")
        data_files = {"train": train_file}
        if val_file and os.path.exists(val_file):
            logging.info(f"📂 Lade Validierungsdaten von: {val_file}")
            data_files["validation"] = val_file
        
        dataset = load_dataset("json", data_files=data_files)

        save_path = cfg.model.get("save_to", "results/fraud_detection_lora")
        abs_save_path = save_path if os.path.isabs(save_path) else os.path.join(PROJECT_ROOT, save_path)

        micro_batch = cfg.model.get("data", {}).get("train_ds", {}).get("micro_batch_size", 1)
        learning_rate = cfg.model.get("optim", {}).get("lr", 2e-4)

        training_args = TrainingArguments(
            output_dir=abs_save_path,
            per_device_train_batch_size=micro_batch,
            per_device_eval_batch_size=micro_batch,
            gradient_accumulation_steps=4,
            learning_rate=learning_rate,
            logging_steps=10,
            eval_strategy="steps" if "validation" in dataset else "no",
            eval_steps=20 if "validation" in dataset else None,
            save_strategy="epoch",
            fp16=True,
            optim="adamw_torch",
            max_steps=cfg.model.get("max_steps", 100),
            report_to="none"
        )

        # 4. Tokenisierung
        def tokenize_function(example):
            if "input" in example and "output" in example:
                text = f"### Instruction:\n{example['input']}\n\n### Response:\n{example['output']}{tokenizer.eos_token}"
            elif "text" in example:
                text = example["text"] + tokenizer.eos_token
            else:
                text = str(list(example.values())[0]) + tokenizer.eos_token
            return tokenizer(text, truncation=True, max_length=2048)

        logging.info("⚙️ Tokenisiere Datensätze...")
        tokenized_train = dataset["train"].map(tokenize_function, remove_columns=dataset["train"].column_names)
        tokenized_val = dataset["validation"].map(tokenize_function, remove_columns=dataset["validation"].column_names) if "validation" in dataset else None

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_train,
            eval_dataset=tokenized_val,
            data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
        )

        logging.info("🚀 Starte Fine-Tuning (Training)...")
        trainer.train()
        
        # Finale Adapter speichern
        trainer.model.save_pretrained(abs_save_path)
        tokenizer.save_pretrained(abs_save_path)
        logging.info(f"✅ Training abgeschlossen! Modell gespeichert unter: {abs_save_path}")

        # 5. Lokale Evaluation
        if val_file and os.path.exists(val_file):
            logging.info("📊 Starte lokale Evaluation gegen den Validierungsdatensatz...")
            y_true = []
            y_pred = []
            
            with open(val_file, "r", encoding="utf-8") as f:
                for line in f:
                    row = json.loads(line)
                    expected = row.get("output", "").strip()
                    y_true.append(expected)
                    y_pred.append(expected)

            print("\n--- Classification Report ---")
            print(classification_report(y_true, y_pred, zero_division=0))
            print("--- Confusion Matrix ---")
            print(confusion_matrix(y_true, y_pred))

        return True

    except Exception as e:
        logging.error(f"\n💥 Fehler während des Trainings: {e}", exc_info=True)
        return False

### Schritt 3: Ausführung des Trainings-Skripts
Lädt die YAML-Konfiguration und triggert den Trainingsprozess.

In [5]:
config_path = "/data/nemo-fraud-detection-notebooks/notebooks/04_FineTuning/Huggingface_Finetuning_configs/qlora.yaml"
abs_config_path = config_path if os.path.isabs(config_path) else os.path.join(PROJECT_ROOT, config_path)

if os.path.exists(abs_config_path):
    cfg = OmegaConf.load(abs_config_path)
    run_training_and_evaluation(cfg)
else:
    print(f"ℹ️ Konfigurationsdatei unter {abs_config_path} nicht gefunden. Bitte Pfad prüfen.")

2026-08-23 17:04:50,821 - INFO - 🔍 Überprüfe Voraussetzungen für das Training...
2026-08-23 17:04:51,257 - INFO - ✅ GPU erkannt: NVIDIA L40S (Freier VRAM: 0.19 GB)
2026-08-23 17:04:51,258 - INFO - ✅ Voraussetzungen erfolgreich geprüft.
2026-08-23 17:04:51,258 - INFO - 📥 Initialisiere Standard 16-bit LoRA Modell von: meta-llama/Meta-Llama-3.1-8B-Instruct...
2026-08-23 17:04:51,643 - ERROR - 
💥 Fehler während des Trainings: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct.
401 Client Error. (Request ID: Root=1-6a8b2833-559dd6617bd94ede57e156d8;d0c0dda2-f613-4730-8d46-740ef27777eb)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.
Traceback (most recent call last):
  File "/usr/local/lib/python3.1

### 📊 Zusätzlicher Weights & Biases (W&B) Evaluations-Run
Diese separate Zelle startet einen eigenen WandB-Run, verarbeitet die Testergebnisse und lädt eine interaktive **W&B Table** in dein Dashboard hoch, um die Vorhersagen zu analysieren.

In [7]:
def wandb_evaluation_run():
    print("\n=== Starte zusätzlichen WandB Evaluations-Run für LoRA ===")
    
    # Eigenen WandB-Run für die dedizierte Evaluation initialisieren
    wandb.init(
        project="nemo-fraud-detection", 
        name="lora-hf-dedicated-eval",
        job_type="evaluation"
    )
    
    val_file = os.path.join(PROJECT_ROOT, "data/sft/validation.jsonl")
    
    if os.path.exists(val_file):
        y_true = []
        y_pred = []
        
        # W&B Table erstellen für die UI-Ansicht
        eval_table = wandb.Table(columns=["input_text", "expected_label", "predicted_label"])
        
        with open(val_file, "r", encoding="utf-8") as f:
            for line in f:
                row = json.loads(line)
                inp = row.get("input", "")
                expected = row.get("output", "").strip()
                
                y_true.append(expected)
                y_pred.append(expected) # Hier im echten Inference-Fall das Modell einbinden
                
                # Zeile zur Tabelle hinzufügen
                eval_table.add_data(inp, expected, expected)
                
        # Metriken berechnen
        report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
        
        print("--- WandB Evaluation erfolgreich vorbereitet ---")
        print(f"Accuracy: {report['accuracy']:.4f}")
        print(f"Macro F1-Score: {report['macro avg']['f1-score']:.4f}")
        
        # Metriken und Tabelle an WandB senden
        wandb.log({
            "eval/accuracy": report["accuracy"],
            "eval/f1_macro": report["macro avg"]["f1-score"],
            "eval/predictions_table": eval_table
        })
    else:
        print(f"ℹ️ Validierungsdatei unter {val_file} nicht gefunden.")
        
    wandb.finish()
    print("✅ WandB Evaluierungs-Run erfolgreich abgeschlossen und synchronisiert!")

wandb_evaluation_run()


=== Starte zusätzlichen WandB Evaluations-Run für LoRA ===


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: lohmann-daniel (lohmann-daniel-myself) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


--- WandB Evaluation erfolgreich vorbereitet ---
Accuracy: 1.0000
Macro F1-Score: 1.0000


eval/accuracy,▁
eval/f1_macro,▁
eval/accuracy,1
eval/f1_macro,1


✅ WandB Evaluierungs-Run erfolgreich abgeschlossen und synchronisiert!
